In [1]:
import os
import shutil
from PIL import Image
from tqdm import tqdm

In [2]:
def get_all_jpeg_files(root_dir):
    """Récupère tous les fichiers JPEG dans root_dir et ses sous-répertoires."""
    jpeg_files = []
    for dirpath, _, filenames in os.walk(root_dir):
        if '-mémoire' in dirpath:  # Exclure les répertoires du genre "Notes et aide-mémoires"
            continue
        if 'très longue' in dirpath:  # Exclure "mariage - version très longue"
            continue
        if 'Allégeance' in dirpath:  # Exclure "Allégeance"
            continue
        if "Visites d'appart" in dirpath:
            continue
        if "2017 05 03 Visite" in dirpath:
            continue
        if "2017 05 14 Pré-état des lieux" in dirpath:
            continue
        if "2011 05 30 Ma chérie" in dirpath:
            continue
        if "Etat des lieux" in dirpath:
            continue
        if "Paris Picpus, dégât des eaux" in dirpath:
            continue
        for file in filenames:
            if file.lower().endswith(('.jpg', '.jpeg')):
                jpeg_files.append(os.path.join(dirpath, file))
    return jpeg_files

In [3]:
def resize_image(image_path):
    """Redimensionne l'image si la plus grande dimension est > 1024 pixels."""
    with Image.open(image_path) as img:
        exif = img._getexif()
        orientation = exif.get(0x0112) if exif else None
        if orientation in [6, 8]:  # 6 = rotation 90°, 8 = rotation 270°
            return  # Do not resize as it would probably mess up orientation
        width, height = img.size
        max_dim = max(width, height)
        if max_dim > 1024:
            if max_dim == width:
                new_width = 1024
                new_height = round(height * new_width / width)
            else:
                new_height = 1024
                new_width = round(width * new_height / height)
            img = img.resize((new_width, new_height), Image.LANCZOS)
            img.save(image_path, quality=95)

In [4]:
def process_images(src_dir, dest_portrait, dest_landscape):
    """Trie et redimensionne les images en fonction de leur orientation et de leur taille."""
    jpeg_files = get_all_jpeg_files(src_dir)
    
    for file_path in tqdm(jpeg_files, desc="Traitement des images"):
        with Image.open(file_path) as img:
            width, height = img.size
            exif = img._getexif()
            orientation = exif.get(0x0112) if exif else None
            if orientation in [6, 8]:  # 6 = rotation 90°, 8 = rotation 270°
                    width, height = height, width
        
        if height > width:
            dest_dir = dest_portrait
        else:
            dest_dir = dest_landscape
        
        os.makedirs(dest_dir, exist_ok=True)
        clean_name = os.path.basename(file_path).replace(':', '_')
        dest_path = os.path.join(dest_dir, clean_name)
        shutil.copy2(file_path, dest_path)

        try:
            resize_image(dest_path)
        except OSError:
            pass

In [5]:
# Noter que ceci NE VIDE PAS src_directory. Le script peut donc être lancé en toute sécurité !
src_directory = r"c:\Dropbox\A\70_Photos_old\9999 A ranger puis basculer vers Google Photos\En traitement\05 Exported jpeg"
dest_portrait = r"c:\Dropbox\A\70_Photos_old\9999 A ranger puis basculer vers Google Photos\En traitement\07 Dispatch vers cadres et répertoires définitifs\v"
dest_landscape = r"c:\Dropbox\A\70_Photos_old\9999 A ranger puis basculer vers Google Photos\En traitement\07 Dispatch vers cadres et répertoires définitifs\h"
process_images(src_directory, dest_portrait, dest_landscape)

Traitement des images: 100%|███████████████████████████████████████████████████████████| 81/81 [00:14<00:00,  5.71it/s]
